# 0.0 - Imports

## 0.1 - Importa as bibliotecas

In [1]:
import os
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

## 0.2 - Carrega as credenciais do .env

In [2]:
load_dotenv()

True

## 0.3 - Conexão utilizando o .env

In [3]:
user = os.getenv('DB_USER')
password = os.getenv('DB_PASS')
host = os.getenv('DB_HOST')
dataset = os.getenv('DB_NAME')

url = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{dataset}")

## 0.4 - Teste de Conexão

In [4]:
with url.connect() as conn:
    result = conn.execute(text("SELECT VERSION();"))
    print(result.fetchone())

('8.0.31-google',)


# 1.0 - Execução das consultas

In [5]:
#Transforma as query em texto para pandas
query_story_cad = text("SELECT STORE_CODE, STORE_NAME, START_DATE, END_DATE, BUSINESS_NAME, BUSINESS_CODE FROM data_store_cad")
query_store_sales = text("SELECT STORE_CODE, DATE, SALES_VALUE, SALES_QTY FROM data_store_sales WHERE DATE BETWEEN '2019-01-01' AND '2019-12-31'")

In [7]:
#Carrega no pandas as duas tabelas
df_raw_cad = pd.read_sql_query(query_story_cad, con=url)
df_raw_sales = pd.read_sql_query (query_store_sales, con=url)

In [8]:
#Verificação dos tipos do DataFrame
print(df_raw_cad.dtypes)
print(df_raw_sales.dtypes)

STORE_CODE       int64
STORE_NAME         str
START_DATE         str
END_DATE           str
BUSINESS_NAME      str
BUSINESS_CODE    int64
dtype: object
STORE_CODE       int64
DATE            object
SALES_VALUE    float64
SALES_QTY        int64
dtype: object


In [9]:
#Verifica se datasets estão de acordo com exemplos
print(df_raw_cad.head(5))
print(df_raw_sales.head(5))

   STORE_CODE STORE_NAME  START_DATE END_DATE BUSINESS_NAME  BUSINESS_CODE
0           1  Sao Paulo  2006-10-01                 Varejo              1
1           2    Chicago  2007-10-01                 Varejo              1
2           3       Roma  2008-10-01                 Varejo              1
3           4      Tokio  2009-10-01                 Varejo              1
4           5      Paris  2019-01-01            Proximidade              2
   STORE_CODE        DATE  SALES_VALUE  SALES_QTY
0           1  2019-01-01    196623.22      12838
1          10  2019-01-01    126795.44       4933
2          11  2019-01-01    223937.00       7724
3          12  2019-01-01    200251.80       7043
4          13  2019-01-01    196623.22      12838


# 2.0 - Joins dos datasets

In [10]:
#Inner Join das duas tabelas
df1 = pd.merge(df_raw_cad, df_raw_sales, on='STORE_CODE', how='inner')
df1.head(5)

,STORE_CODE,STORE_NAME,START_DATE,END_DATE,BUSINESS_NAME,BUSINESS_CODE,DATE,SALES_VALUE,SALES_QTY
0,1,Sao Paulo,2006-10-01,,Varejo,1,2019-01-01,196623.22,12838
1,1,Sao Paulo,2006-10-01,,Varejo,1,2019-01-02,307504.48,19973
2,1,Sao Paulo,2006-10-01,,Varejo,1,2019-01-03,274842.91,17860
3,1,Sao Paulo,2006-10-01,,Varejo,1,2019-01-04,242439.39,15713
4,1,Sao Paulo,2006-10-01,,Varejo,1,2019-01-05,205547.18,13338


In [11]:
#Cria uma cópia do Dataset Final (original a partir desse ponto)
df2 = df1.copy()

In [12]:
#Transforma coluna DATE de objetct para datetime64
df2['DATE'] = pd.to_datetime(df2['DATE'])
df2.dtypes

STORE_CODE               int64
STORE_NAME                 str
START_DATE                 str
END_DATE                   str
BUSINESS_NAME              str
BUSINESS_CODE            int64
DATE             datetime64[s]
SALES_VALUE            float64
SALES_QTY                int64
dtype: object

In [13]:
#Filtra o dataset conforme condição: "Filtre o período dentro do intervalo fornecido: ['2019-10-01','2019-12-31']"
data_inicio = pd.to_datetime('2019-10-01')
data_fim = pd.to_datetime('2019-12-31')

df2 = df2[(df2['DATE'] >= data_inicio) & (df2['DATE'] <= data_fim)]

#Verifica a aplicação do filtro
date_min = df2['DATE'].min()
print(date_min)

date_max = df2['DATE'].max()
print(date_max)

2019-10-01 00:00:00
2019-12-31 00:00:00


In [14]:
#Ajuste das colunas que vão ficar no dataset para analise do TM
cols_df = ['STORE_NAME', 'BUSINESS_NAME', 'SALES_VALUE', 'SALES_QTY']

df2 = df2[cols_df]
df2.head(5)

,STORE_NAME,BUSINESS_NAME,SALES_VALUE,SALES_QTY
273,Sao Paulo,Varejo,187601.54,12160
274,Sao Paulo,Varejo,332666.14,21643
275,Sao Paulo,Varejo,253401.57,16489
276,Sao Paulo,Varejo,257709.59,16724
277,Sao Paulo,Varejo,216934.36,14120


# 3.0 - Calculo do Ticket Médio

In [15]:
# Copia para manipulação do dataset
df3 = df2.copy()

In [16]:
# Agrupa por loja
df_tm = df3.groupby(['STORE_NAME','BUSINESS_NAME'])[['SALES_VALUE', 'SALES_QTY']].sum().reset_index()
df_tm

,STORE_NAME,BUSINESS_NAME,SALES_VALUE,SALES_QTY
0,Bahia,Atacado,21213088.57,1378476
1,Bangkok,Posto,8376271.00,612968
2,Belem,Proximidade,20989553.37,1365988
3,Berlin,Proximidade,21213088.57,1378476
4,Buenos Aires,Atacado,21213088.57,1378476
5,Chicago,Varejo,21928421.28,1412372
6,Dubai,Atacado,21213088.57,1378476
7,Hong Kong,Farma,15039911.54,570745
8,London,Farma,19471788.15,671638
9,Madri,Farma,24129399.10,831168


In [17]:
#Calcula Ticket Médio
df_tm['TM'] = round(df_tm['SALES_VALUE'] / df_tm['SALES_QTY'], 2)

In [18]:
#Ajusta colunas que vão ficar no dataset finak
df_tm = df_tm[['STORE_NAME', 'BUSINESS_NAME', 'TM']]

#Renomeia colunas conforme tabela do case
df_tm = df_tm.rename(columns={
    'STORE_NAME':       'Lojas',
    'BUSINESS_NAME' :   'Categoria'
})

# 4.0 - Tabela Final

In [19]:
df_tm

,Lojas,Categoria,TM
0,Bahia,Atacado,15.39
1,Bangkok,Posto,13.67
2,Belem,Proximidade,15.37
3,Berlin,Proximidade,15.39
4,Buenos Aires,Atacado,15.39
5,Chicago,Varejo,15.53
6,Dubai,Atacado,15.39
7,Hong Kong,Farma,26.35
8,London,Farma,28.99
9,Madri,Farma,29.03
